In [55]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta, domain_best_by_model, recommended_by_domain_for_model
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)
test_df = pd.read_csv(paths.TEST_PATH)

exp_configs = ALL_EXPERIMENTS["fe12__sex_pclass"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     print(f"Experiment config: {exp_config}")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True, context_df=test_df)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size
Experiment: fe10__fare_per_ticket_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_pclass_and_bins
Experiment: cb04__fare_and_fare_per_family
Experiment: cb05__fare_and_fare_per_ticket
Experiment: cb06__all_fare_features
Experiment: cb07__family_features
Experiment: cb08__sex_pclass_features
Experiment: ab01__age_and_bins_without_fare
Experiment: ab02__age_imputed_title_and_bins_without_fare
Experiment: ab03__age_imputed_title_pclass_and_bins_without_fare
Experiment: ab04__age_bin_without_fare
Experiment: ab05__sex_pclass_without_sibsp_parch
Experi

In [57]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [58]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
    context_df=test_df,
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

In [59]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results()
references = ['baseline__raw']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
#### fe12__sex_pclass

_Description pending._

<details>
<summary>Conclusion</summary>


##### Interpretation

- Verdict: mixed
- Recommended for specific models:
  - logreg: test_accuracy_mean: 0.016
  - extra_trees: test_accuracy_mean: 0.006
    - Secondary losses:
      - test_f1_mean: -0.019


##### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

##### Comparison vs baseline__raw

| reference_group   | compare_group    | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:-----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe12__sex_pclass | logreg        |                          0.

In [60]:
# raise to quickly run small experiments and reports
raise

RuntimeError: No active exception to reraise

In [ ]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=10, selection="all")
print("Current leaderboard:")
print(current_leaderboard)

Current leaderboard:
| experiment                                                          | model_name    |   test_accuracy_mean |   test_f1_mean |
|:--------------------------------------------------------------------|:--------------|---------------------:|---------------:|
| fe05__title__xgb                                                    | xgb           |                0.836 |          0.772 |
| ab02__age_imputed_title_and_bins_without_fare__xgb                  | xgb           |                0.835 |          0.766 |
| cb05__fare_and_fare_per_ticket_full_context__xgb                    | xgb           |                0.834 |          0.772 |
| fe05__title__svc                                                    | svc           |                0.834 |          0.771 |
| ab03__age_imputed_title_Pclass_and_bins_without_fare__random_forest | random_forest |                0.834 |          0.762 |
| ab03__age_imputed_title_pclass_and_bins_without_fare__random_forest | random_fore

In [ ]:
from titanic_ml.common.experiments.inspection import get_feature_pipeline, get_cv_fold, inspect_experiment_fold
from titanic_ml.common.experiments.runner import build_model_pipeline
for exp_name, exp_config in exp_configs.items():
    exp = exp_config
    break

model_pipeline = build_model_pipeline(exp)
for name, step in model_pipeline.steps:
    print(name, type(step).__name__)

family_size SumColumnsTransformer
fare_per_familysize FeatureDividedByFeatureTransformer
fare_per_ticket FeatureDividedByFeatureTransformer
preprocessor ColumnTransformer
model LogisticRegression


In [ ]:
inspection = inspect_experiment_fold(
    df=train_df,
    experiments=ALL_EXPERIMENTS["test_family_ticket_full_context_config"],
    target="Survived",
    context_df=test_df,
    fold=0,
)

c:\Users\jonma\Documents\GitHub\Machine_learning\Titanic\.venv\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [ ]:
inspection[
    "X_validation_engineered"
].columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'TicketGroupSize', 'FamilySize',
       'Fare/FamilySize', 'Fare/TicketMember'],
      dtype='object')

In [ ]:
train = inspection["X_train_cv_input"]
validation = inspection[
    "X_validation_engineered"
]

row = validation.iloc[0]

ticket = row["Ticket"]

print(
    "Ticket:",
    ticket,
)

print(
    "Training count:",
    train["Ticket"].eq(ticket).sum(),
)

print(
    "Engineered count:",
    row["TicketGroupSize"],
)

Ticket: A/5 21171
Training count: 0
Engineered count: 1.0


In [ ]:
validation_raw = inspection[
    "X_train_cv_input"
]

validation = inspection[
    "X_validation_engineered"
]

row = validation.iloc[0]
ticket = row["Ticket"]

print(
    validation_raw["Ticket"]
    .eq(ticket)
    .sum()
)

print(
    row["TicketGroupSize"]
)

0
1.0


In [ ]:
inspection["X_train_cv_input"]
inspection["X_validation_cv_input"]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,TicketGroupSize
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1.0
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2.0
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1.0
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2.0
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
192,193,3,"Andersen-Jensen, Miss. Carla Christine Nielsine",female,19.0,1,0,350046,7.8542,NaN,S,1.0
193,194,2,"Navratil, Master. Michel M",male,3.0,1,1,230080,26.0000,F2,S,3.0
194,195,1,"Brown, Mrs. James Joseph (Margaret Tobin)",female,44.0,0,0,PC 17610,27.7208,B4,C,1.0
195,196,1,"Lurette, Miss. Elise",female,58.0,0,0,PC 17569,146.5208,B80,C,3.0


In [ ]:
df = inspection[
    "X_validation_engineered"
]

(
    df["FamilySize"]
    ==
    df["SibSp"] + df["Parch"] + 1
).all()

np.True_

In [ ]:
def compare_group_representations(
    df,
    family_col="FamilySize",
    ticket_col="TicketGroupSize",
    family_fare_col="Fare/FamilySize",
    ticket_fare_col="Fare/TicketMember",
):
    same_group_size = df[family_col].eq(df[ticket_col])

    return {
        "group_size_equal_rate": same_group_size.mean(),

        "group_size_mae": (
            df[family_col] - df[ticket_col]
        ).abs().mean(),

        "fare_pearson": df[
            [family_fare_col, ticket_fare_col]
        ].corr(method="pearson").iloc[0, 1],

        "fare_spearman": df[
            [family_fare_col, ticket_fare_col]
        ].corr(method="spearman").iloc[0, 1],

    "family_unique": df[family_fare_col].nunique(),

    "ticket_unique": df[ticket_fare_col].nunique(),
    
    "exact_fare_equal_rate": (
        df[family_fare_col] == df[ticket_fare_col]
    ).mean(),
}


In [ ]:
print("Comparing family and ticket group representations:")
compare_group = compare_group_representations(inspection["X_validation_engineered"])
for key, value in compare_group.items():
    print(f"{key}: {value}")

Comparing family and ticket group representations:
group_size_equal_rate: 0.8268156424581006
group_size_mae: 0.30726256983240224
fare_pearson: 0.8402022150479401
fare_spearman: 0.8659111226162234
family_unique: 104
ticket_unique: 93
exact_fare_equal_rate: 0.8268156424581006


In [ ]:
raise

RuntimeError: No active exception to reraise

In [ ]:
# thresholds = {
#     "accuracy": 0.003,
#     "f1": -0.01,
# }

# for model in MODEL_REGISTRY:

#     result = recommended_by_domain_for_model(
#         results_df=all_results,
#         model_name=model,
#         thresholds=thresholds,
#     )

#     print(f"\nBest candidates for {model}")
#     print("=" * 50)

#     print("\nRecommended:")
#     for recommendation in result["recommended"]:
#         print(
#             recommendation["domain"],
#             "->",
#             recommendation["group"],
#             recommendation["deltas"],
#         )
#     print()
#     print("recommended list:")
#     for recommendation in result["recommended"]:
#         print(recommendation["group"], end=", ")
#     print()
#     print("\nFull domain results:")
#     print(result["df"].to_markdown(index=False))

In [ ]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [ ]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [ ]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [ ]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [ ]:
# print(workflow["all_results"])

In [ ]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

Old Eda practice

In [ ]:
# eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print(eda)

In [ ]:
# print(eda)

In [ ]:
# print(eda['dataFrame_health'].to_markdown())

In [ ]:
# print(eda["dataFrame_summary"].to_markdown())

In [ ]:
# for col in eda['categorical_summary']:
#     sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
#     sample_df = pd.concat(
#         [sample[k] for k in ['head', 'random', 'tail']],
#         ignore_index=False
#     )
#     print(sample_df.to_markdown())
#     print()

In [ ]:
# for col in eda['numerical_summary']:
#     print(f"Numerical column: {col}")
#     print(eda['numerical_summary'][col].to_markdown())
#     print()

In [ ]:
# print("Correlation matrix:")
# print(eda["correlation_matrix"].to_markdown())

In [ ]:
# print('Correlation with the target variable:')
# print(eda["target_correlation"].to_markdown())

In [ ]:
# for col in eda["categorical_rare"]:
#     print(f"Categorical column with rare values: {col}")
#     print(eda["categorical_rare"][col])
#     print()

In [ ]:
# # print(eda['sample'])
# sample_df = pd.concat(
#     [eda['sample'][k] for k in ['head', 'random', 'tail']],
#     ignore_index=True
# )
# print(sample_df.to_markdown(index=False))

In [ ]:
# Code to test the updates to runner

from sklearn.base import BaseEstimator, TransformerMixin


class ContextPopulationSizeTransformer(
    BaseEstimator,
    TransformerMixin,
):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["ContextPopulationSize"] = len(X)

        return X

In [ ]:
TEST_PRE_CV_POPULATION_SIZE = {
    "id": "context_population_size",
    "stage": "pre_cv",
    "transformer": ContextPopulationSizeTransformer,
    "requires": [],
    "produces": ["ContextPopulationSize"],
    "owns": ["ContextPopulationSize"],
}

In [ ]:
TEST_PRE_CV_PATCH = {
    "pre_cv_transformations": [
        TEST_PRE_CV_POPULATION_SIZE,
    ],
    "add": {
        "preprocessing": {
            "numeric_features": [
                "ContextPopulationSize",
            ],
        },
    },
}

In [ ]:
from titanic_ml.common.experiments.config_creation import create_config
from titanic_ml.common.experiments.config import RAW_FEATURES

test_config = create_config(
    base_config=ALL_EXPERIMENTS["baseline__raw"]["logreg__raw"],
    patches=[TEST_PRE_CV_PATCH],
    raw_features=RAW_FEATURES,
    stage="test",
    feature_group="pre_cv",
    domain="test",
    notes="Test config to test the updated pipeline",
    pre_cv_scope="full_prediction_context",
)

In [ ]:
for config in test_config.items():
    print(config)

('pre_cv_feature_pipeline', [('context_population_size', ContextPopulationSizeTransformer())])
('pre_cv_scope', 'full_prediction_context')
('preprocessing', {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare', 'ContextPopulationSize'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'})
('feature_pipeline', [])
('evaluation', {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1})
('stage', 'test')
('feature_group', 'pre_cv')
('group', 'test__pre_cv')
('domain', 'test')
('tags', set())
('model_name', 'logreg')
('model_params', {'max_iter': 1000, 'random_state': 42})
('notes', 'Test config to test the updated pipeline')
('name', 'test__pre_cv__logreg')
('features', ['Age', 'SibSp', 'Parch', 'Fare', 'ContextPopulationSize', 'Sex', 'Embarked', 'Pclass'])


In [ ]:
from titanic_ml.common.experiments.runner import apply_pre_cv_feature_pipeline 
X = train_df.drop(
    columns=["Survived"]
)

X_pre_cv = apply_pre_cv_feature_pipeline(
    X=X,
    exp=test_config,
    context_df=test_df,
)

In [ ]:
assert len(X_pre_cv) == len(X)

assert X_pre_cv.index.equals(X.index)

assert (
    X_pre_cv["ContextPopulationSize"]
    == len(X) + len(test_df)
).all()

assert "ContextPopulationSize" not in X.columns

In [ ]:
assert X_pre_cv["PassengerId"].equals(
    X["PassengerId"]
)

In [ ]:
small_context = test_df.iloc[:10]

X_small_context = apply_pre_cv_feature_pipeline(
    X=X,
    exp=test_config,
    context_df=small_context,
)

assert (
    X_small_context["ContextPopulationSize"]
    == len(X) + 10
).all()

In [ ]:
# Meant to fail
apply_pre_cv_feature_pipeline(
    X=X,
    exp=test_config,
    context_df=None,
)

ValueError: Experiment uses pre-CV scope 'full_prediction_context', but no context_df was provided.